In [1]:
import pandas as pd
import torch
import random

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments


df = pd.read_csv("arab_meals_120.csv")

df.columns = df.columns.str.strip()

print(df.head())

c:\Users\IFIX\Documents\university\مشروع تخرج\ml-model-chat\ml-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


          Goal  Meal Type                                               Name  \
0  Muscle Gain  Breakfast  Foul Medames with Olive Oil and Whole Wheat Bread   
1     Fat Loss  Breakfast                Greek Yogurt with Dates and Walnuts   
2  Maintenance  Breakfast                   Shakshuka with Whole Wheat Bread   
3  Muscle Gain  Breakfast                  Labneh with Za'atar and Olive Oil   
4    Endurance  Breakfast                     Oatmeal with Tahini and Banana   

   Calories  ProteinContent  CarbohydrateContent  FatContent  \
0       430              18                   48          16   
1       320              20                   28          12   
2       410              22                   30          18   
3       390              17                   24          22   
4       450              16                   60          14   

                                         Ingredients         Cuisine  
0  fava beans, olive oil, lemon, garlic, whole wh...  Middle Ea

Intent Dataset (Synthetic Training Data)

In [2]:
intents = [
    ("I want a meal for muscle gain", "Recommend a meal"),
    ("Suggest breakfast for fat loss", "Recommend a meal"),
    ("What should I eat after workout?", "Recommend a post-workout meal"),
    ("What is good before gym?", "Recommend a pre-workout meal"),
    ("Analyze Shakshuka", "Analyze a meal"),
    ("Compare chicken and hummus meals", "Compare meals"),
    ("Give me healthy meals", "List healthy meals"),
    ("Create a daily meal plan", "Create a daily meal plan"),
    ("Create a weekly meal plan", "Create a weekly meal plan"),
    ("Suggest substitution for bread", "Suggest a substitution"),
]

Labels Encoding

In [3]:
labels = list(set([i[1] for i in intents]))
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

X = [i[0] for i in intents]
y = [label2id[i[1]] for i in intents]

BERT Tokenizer

In [4]:
model_name = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

encodings = tokenizer(X, truncation=True, padding=True)

Dataset Class

In [5]:
class MealDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

Train Intent Model (BERT)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    logging_steps=10,
    learning_rate=2e-5
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=MealDataset(encodings, y)
)

trainer.train()

Predict Intent

In [ ]:
def predict_intent(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=1).item()

    return id2label[prediction]

Meal Recommendation Engine

In [ ]:
def recommend_meal(goal, meal_type):
    results = df[
        (df["Goal"].str.lower() == goal.lower()) &
        (df["Meal Type"].str.lower() == meal_type.lower())
    ]

    if results.empty:
        return "No meals found"

    return results.sample(1).iloc[0].to_dict()

Meal Analysis

In [ ]:
def analyze_meal(name):
    meal = df[df["Name"].str.lower() == name.lower()]

    if meal.empty:
        return "Meal not found"

    return meal.iloc[0].to_dict()

Meal Comparison

In [ ]:
def compare_meals(m1, m2):
    meal1 = df[df["Name"].str.contains(m1, case=False)]
    meal2 = df[df["Name"].str.contains(m2, case=False)]

    if meal1.empty or meal2.empty:
        return "One or both meals not found"

    return {
        "Meal 1": meal1.iloc[0].to_dict(),
        "Meal 2": meal2.iloc[0].to_dict()
    }

Daily Meal Plan

In [ ]:
def create_daily_plan(goal):
    breakfast = df[(df["Goal"] == goal) & (df["Meal Type"] == "Breakfast")]
    lunch = df[(df["Goal"] == goal) & (df["Meal Type"] == "Lunch")]
    dinner = df[(df["Goal"] == goal) & (df["Meal Type"] == "Dinner")]

    return {
        "Breakfast": breakfast.sample(1).iloc[0].to_dict() if not breakfast.empty else None,
        "Lunch": lunch.sample(1).iloc[0].to_dict() if not lunch.empty else None,
        "Dinner": dinner.sample(1).iloc[0].to_dict() if not dinner.empty else None
    }

Weekly Meal Plan

In [ ]:
def create_weekly_plan(goal):
    days = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

    return {day: create_daily_plan(goal) for day in days}

Chatbot Controller (Core System)

In [ ]:
def chatbot(user_input):
    intent = predict_intent(user_input)

    if intent == "Recommend a meal":
        return recommend_meal("Muscle Gain", "Breakfast")

    elif intent == "Analyze a meal":
        return analyze_meal(user_input)

    elif intent == "Compare meals":
        return compare_meals("Shakshuka", "Omelette")

    elif intent == "List healthy meals":
        return df.sample(5).to_dict()

    elif intent == "Create a daily meal plan":
        return create_daily_plan("Muscle Gain")

    elif intent == "Create a weekly meal plan":
        return create_weekly_plan("Muscle Gain")

    elif intent == "Recommend a pre-workout meal":
        return recommend_meal("Endurance", "Breakfast")

    elif intent == "Recommend a post-workout meal":
        return recommend_meal("Muscle Gain", "Breakfast")

    else:
        return "Sorry, I can help with nutrition and meal planning only."

Testing

In [ ]:
print(chatbot("I want something for muscle gain"))
print(chatbot("Analyze Shakshuka"))
print(chatbot("Compare meals"))
print(chatbot("Create a weekly meal plan"))